# Exercise 2 — Neural networks

This exercise accompanies **Lecture 2 (Oct 20), Sessions 2–4**. You will:

- implement a **one-hidden-layer MLP from scratch** in numpy — forward pass, backpropagation and gradient descent,
- compare your network's decision boundary against `MLPClassifier`,
- explore how **capacity** (hidden units) affects test accuracy,
- apply `MLPClassifier`/`MLPRegressor` to a **diverse set of datasets**: binary, multi-class and regression.

**Datasets:** `make_circles`, `make_moons`, `load_digits`, `load_wine`,
`load_diabetes` — all bundled with scikit-learn.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles, make_moons, load_digits, load_wine, load_diabetes
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    r2_score,
    mean_squared_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

## Part A — A one-hidden-layer MLP from scratch (numpy)

Network: **2 inputs → 8 hidden units (tanh) → 1 output (sigmoid)**, trained
with full-batch gradient descent on the binary cross-entropy.

**Forward pass** (shapes for $n$ samples):

| step | formula |
|---|---|
| hidden pre-activation | $Z_1 = XW_1 + b_1$ — shape $(n, 8)$ |
| hidden activation | $A_1 = \tanh(Z_1)$ |
| output pre-activation | $Z_2 = A_1 W_2 + b_2$ — shape $(n, 1)$ |
| probability | $\hat{p} = \sigma(Z_2)$ |

**Backward pass** (chain rule; the $1/n$ comes from the *mean* loss):

$$\delta_2 = \frac{\hat{p} - y}{n} \quad (n,1)$$

$$\frac{\partial \mathcal{L}}{\partial W_2} = A_1^\top \delta_2, \qquad
\frac{\partial \mathcal{L}}{\partial b_2} = \sum_i \delta_{2,i}$$

$$\delta_1 = (\delta_2 W_2^\top) \odot (1 - A_1^2) \quad (n,8)$$

$$\frac{\partial \mathcal{L}}{\partial W_1} = X^\top \delta_1, \qquad
\frac{\partial \mathcal{L}}{\partial b_1} = \sum_i \delta_{1,i}$$

### A.1 — Concentric circles (standardised)

In [ ]:
X_c, y_c = make_circles(n_samples=400, factor=0.4, noise=0.1, random_state=0)
scaler = StandardScaler().fit(X_c)
X_cs = scaler.transform(X_c)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_cs[y_c == 0, 0], X_cs[y_c == 0, 1], color="#dc2626", s=16, label="class 0")
ax.scatter(X_cs[y_c == 1, 0], X_cs[y_c == 1, 1], color="#16a34a", s=16, label="class 1")
ax.set_aspect("equal")
ax.legend()
ax.set_title("concentric circles — no line will ever fit")
plt.show()

X_tr, X_te, y_tr, y_te = train_test_split(X_cs, y_c, test_size=0.3, random_state=0, stratify=y_c)

### A.2 — Initialise the weights

Small random weights (Xavier-style scaling); biases start at zero.

In [ ]:
rng = np.random.default_rng(0)
n_feat, n_hidden = 2, 8

W1 = rng.normal(scale=np.sqrt(2 / (n_feat + n_hidden)), size=(n_feat, n_hidden))
b1 = np.zeros(n_hidden)
W2 = rng.normal(scale=np.sqrt(2 / (n_hidden + 1)), size=(n_hidden, 1))
b2 = np.zeros(1)

### A.3 — Forward pass

Fill in the function; return the pre-activations/activations so the backward
pass can reuse them.

In [ ]:
def forward(X, W1, b1, W2, b2):
    # YOUR CODE HERE
    ...
    return Z1, A1, Z2


Z1, A1, Z2 = forward(X_tr, W1, b1, W2, b2)
print(f"Z1: {Z1.shape}, A1: {A1.shape}, Z2: {Z2.shape}")
# Expected: (n_train, 8), (n_train, 8), (n_train, 1)

### A.4 — Backward pass + training loop

Fill in the six gradient lines using the equations above. The training loop is
written for you.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


lr, n_iter = 0.5, 3000
n = len(y_tr)
losses = []

for i in range(n_iter):
    Z1, A1, Z2 = forward(X_tr, W1, b1, W2, b2)
    p = sigmoid(Z2).ravel()
    losses.append(
        -np.mean(y_tr * np.log(p + 1e-12) + (1 - y_tr) * np.log(1 - p + 1e-12))
    )

    # YOUR CODE HERE — backpropagation:
    # delta2 = ...            # (n, 1)  output error, scaled by 1/n
    # dW2    = ...            # (8, 1)
    # db2    = ...            # (1,)
    # delta1 = ...            # (n, 8)  through the tanh derivative (1 - A1**2)
    # dW1    = ...            # (2, 8)
    # db1    = ...            # (8,)

    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("cross-entropy")
plt.title("Training loss")
plt.show()
# Expected: the loss drops from ≈ 0.7 towards < 0.1

### A.5 — Accuracy and the learned boundary

Compute the training and test accuracy of your network (threshold $\hat p$ at
0.5), then run the scaffold to draw its decision boundary.

In [ ]:
# YOUR CODE HERE: train and test accuracy of your hand-written network

# Decision boundary (given):
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 250), np.linspace(-2.5, 2.5, 250))
_, _, grid_Z2 = forward(np.c_[xx.ravel(), yy.ravel()], W1, b1, W2, b2)
grid_p = sigmoid(grid_Z2).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.contourf(xx, yy, grid_p, levels=20, cmap="RdYlGn", alpha=0.7)
ax.scatter(X_cs[y_c == 0, 0], X_cs[y_c == 0, 1], color="#dc2626", s=14)
ax.scatter(X_cs[y_c == 1, 0], X_cs[y_c == 1, 1], color="#16a34a", s=14)
ax.set_aspect("equal")
ax.set_title("your MLP — a round boundary from 8 neurons")
plt.show()

### A.6 — Compare against `MLPClassifier`

Same architecture (8 tanh hidden units), but sklearn uses the **Adam**
optimiser. Do the boundaries coincide? Why do the accuracies differ slightly
(hint: optimiser, initialisation, regularisation)?

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", max_iter=3000, random_state=0)
mlp.fit(X_tr, y_tr)

print("hand-written:  test acc = ...")   # YOUR CODE HERE: fill in your value
print(f"MLPClassifier: test acc = {mlp.score(X_te, y_te):.3f}")

## Part B — Capacity: how many hidden units?

### B.1 — Noisy moons

In [ ]:
X_mn, y_mn = make_moons(n_samples=500, noise=0.35, random_state=0)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_mn, y_mn, test_size=0.3, random_state=0, stratify=y_mn)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(X_mn[:, 0], X_mn[:, 1], c=y_mn, cmap="RdYlGn", s=14)
ax.set_aspect("equal")
plt.show()

### B.2 — Accuracy vs architecture

Fit an `MLPClassifier` for each architecture below and record the test
accuracy. Which one wins — and is the biggest model necessarily the best?

In [ ]:
architectures = [(2,), (4,), (8,), (16,), (32,), (32, 32)]
test_accs = []
for hidden in architectures:
    clf = MLPClassifier(hidden_layer_sizes=hidden, max_iter=3000, random_state=0)
    # YOUR CODE HERE: fit on the training set, append the test accuracy
    ...

# YOUR CODE HERE: bar chart of test accuracy per architecture
...

### B.3 — Loss curves

Plot `clf.loss_curve_` for the `(2,)` and `(32, 32)` models. Which converges
faster? Can you spot overfitting in a training loss curve alone — or do you
need a validation set?

In [ ]:
# YOUR CODE HERE
...

## Part C — Multi-class classification on diverse datasets

### C.1 — Handwritten digits (pixels already scaled to [0, 1])

In [ ]:
digits = load_digits()
X_d, y_d = digits.data / 16.0, digits.target

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lbl in zip(axes, digits.images[:8], y_d[:8]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lbl))
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_d, y_d, test_size=0.3, random_state=0, stratify=y_d)

### C.2 — Train an MLP

Fit `MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=300, random_state=0)`
and report the test accuracy (expected ≈ 0.96–0.98).

In [ ]:
# YOUR CODE HERE
...

### C.3 — Hunt the errors

Show the confusion matrix. Which pairs of digits are most confused? Plot a few
misclassified digits with their true and predicted labels — can *you* classify
them correctly?

In [ ]:
# YOUR CODE HERE: confusion matrix + a gallery of misclassified digits
...

### C.4 — Another multi-class set: wine

`load_wine()` has 178 samples, 13 chemical features and 3 grape varieties.
Scale the features (`StandardScaler`), fit an `MLPClassifier`, and compare its
test accuracy with `LogisticRegression`.

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target

# YOUR CODE HERE: split 70/30 (stratified, random_state=0), scale,
# fit MLPClassifier and LogisticRegression, compare test accuracies
...

## Part D — A regression network (diabetes)

Neural networks are not only classifiers. `MLPRegressor` predicts a
**continuous** target — just like the linear regression from Lecture 1.

`load_diabetes()` has 442 patients and 10 features. Scale the features, fit an
`MLPRegressor`, and report test **R²** and **RMSE**. Then compare with the
`LinearRegression` baseline.

In [ ]:
diab = load_diabetes()
X_r, y_r = diab.data, diab.target

# YOUR CODE HERE: split 70/30 (random_state=0), scale the features,
# fit MLPRegressor(hidden_layer_sizes=(32,), max_iter=500, random_state=0)
# and LinearRegression, and report test R² and RMSE for both
...

## Part E — (Bonus) learning-rate sensitivity

`MLPClassifier(learning_rate_init=...)` with architecture `(16,)` on the moons
data: try `[0.001, 0.01, 0.1, 1.0]`, plot the four loss curves, and answer —
which rates converge, which oscillate or diverge?

In [ ]:
# YOUR CODE HERE
...

---

That's the end of the exercise series — you implemented **linear regression**,
**logistic regression** and a **neural network** by hand, and benchmarked
**trees, forests and MLPs** against scikit-learn. Well done!